In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS project_20b;

In [0]:
%sql
USE SCHEMA project_20b;

In [0]:
%sql
SELECT CURRENT_CATALOG(),CURRENT_SCHEMA();

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS project_20b_data;

In [0]:
%sql
SELECT * FROM read_files(
    '/Volumes/workspace/project_20b/project_20b_data/raw_branches.json',
    FORMAT => 'json'
);

In [0]:
%sql
CREATE OR REPLACE TABLE STG_RATINGS
(
    RATING_ID STRING,
    AGENCY STRING,
    CREDIT_SCORE_BAND STRING,
    RISK_TIER STRING
);

In [0]:
%sql
CREATE OR REPLACE TABLE STG_BRANCHES
(
    BRANCH_ID STRING,
    BRANCH_NAME STRING,
    REGION STRING
);

In [0]:
%sql
INSERT INTO STG_RATINGS(RATING_ID,AGENCY,CREDIT_SCORE_BAND,RISK_TIER)
SELECT RATING_ID,AGENCY,CREDIT_SCORE_BAND,RISK_TIER
FROM read_files(
    '/Volumes/workspace/project_20b/project_20b_data/raw_ratings.json',
    FORMAT => 'json'
);

In [0]:
%sql
INSERT INTO STG_BRANCHES(BRANCH_ID,BRANCH_NAME,REGION)
SELECT BRANCH_ID,BRANCH_NAME,REGION
FROM read_files(
    '/Volumes/workspace/project_20b/project_20b_data/raw_branches.json',
    FORMAT => 'json'
);

In [0]:
%sql
SELECT  'STG_RATINGS' AS STAGE_TABLE,
        COUNT(*) AS RECORD_COUNT
FROM STG_RATINGS
UNION ALL
SELECT  'STG_BRANCHES' AS STAGE_TABLE,
        COUNT(*) AS RECORD_COUNT
FROM STG_BRANCHES;

In [0]:
%sql
CREATE OR REPLACE TABLE FACT_CUSTOMER_FACILITY_ELIGIBILITY
(
    CUSTOMER_ID STRING,
    FACILITY_ID STRING,
    ELIGIBILITY_FLAG BOOLEAN
);

In [0]:
%sql
INSERT INTO FACT_CUSTOMER_FACILITY_ELIGIBILITY(CUSTOMER_ID,FACILITY_ID,ELIGIBILITY_FLAG)
SELECT  CUSTOMER_ID,
        FACILITY_ID,
        ELIGIBILITY_FLAG
FROM read_files(
    '/Volumes/workspace/project_20b/project_20b_data/raw_facilities.json',
    FORMAT => 'json'
);

In [0]:
%sql
SELECT * FROM FACT_CUSTOMER_FACILITY_ELIGIBILITY;

In [0]:
%sql
SELECT * FROM read_files(
    '/Volumes/workspace/project_20b/project_20b_data/raw_loan_applications.json',
    FORMAT => 'json'
);

In [0]:
%sql
CREATE TABLE RAW_CUSTOMERS
(
    CUSTOMER_ID STRING,
    CUSTOMER_NAME STRING,
    RATING_ID STRING,
    BRANCH_ID STRING
);

In [0]:
%sql
INSERT INTO RAW_CUSTOMERS(CUSTOMER_ID,CUSTOMER_NAME,RATING_ID,BRANCH_iD)
SELECT  CUSTOMER_ID,
        CUSTOMER_NAME,
        RATING_ID,
        BRANCH_ID
FROM read_files(
    '/Volumes/workspace/project_20b/project_20b_data/raw_customers.json',
    FORMAT => 'json'
);

In [0]:
%sql
CREATE OR REPLACE TABLE FACT_LOAN_APPLICATION_SNAPSHOT
(
    APP_ID STRING,
    CUSTOMER_ID STRING,
    SUBMITTED_DATE DATE,
    UNDERWRITTEN_DATE DATE,
    APPROVED_DATE DATE,
    DISBURSED_DATE DATE,
    LOAN_AMOUNT DECIMAL(10,2)
);

In [0]:
%sql
INSERT INTO FACT_LOAN_APPLICATION_SNAPSHOT(
                                            APP_ID,
                                            CUSTOMER_ID,
                                            SUBMITTED_DATE,
                                            UNDERWRITTEN_DATE,
                                            APPROVED_DATE,
                                            DISBURSED_DATE,
                                            LOAN_AMOUNT
)
SELECT  LOAN.APP_ID,
        LOAN.CUSTOMER_ID,
        LOAN.SUBMITTED_DATE,
        LOAN.UNDERWRITTEN_DATE,
        LOAN.APPROVED_DATE,
        LOAN.DISBURSED_DATE,
        LOAN.LOAN_AMOUNT
FROM read_files(
    '/Volumes/workspace/project_20b/project_20b_data/raw_loan_applications.json',
    FORMAT => 'json'
) LOAN;

In [0]:
%sql
SELECT * FROM FACT_LOAN_APPLICATION_SNAPSHOT;

In [0]:
%sql
SELECT  LOAN.APP_ID,
        CUST.CUSTOMER_NAME,
        BRNCH.BRANCH_NAME,
        RAT.CREDIT_SCORE_BAND,
        RAT.RISK_TIER,
        LOAN.LOAN_AMOUNT
FROM FACT_LOAN_APPLICATION_SNAPSHOT LOAN
JOIN RAW_CUSTOMERS CUST
ON LOAN.CUSTOMER_ID = CUST.CUSTOMER_ID
JOIN STG_BRANCHES BRNCH
ON CUST.BRANCH_ID=BRNCH.BRANCH_ID
JOIN STG_RATINGS RAT
ON CUST.RATING_ID=RAT.RATING_ID

In [0]:
%sql
CREATE TABLE CDC_UPDATES
(
    APP_ID STRING,
    APPROVED_DATE DATE,
    DISBURSED_DATE DATE
);

In [0]:
%sql
INSERT INTO CDC_UPDATES
SELECT  APP_ID,
        APPROVED_DATE,
        DISBURSED_DATE
FROM read_files(
    '/Volumes/workspace/project_20b/project_20b_data/cdc_loan_updates.json',
    FORMAT => 'json'
);

In [0]:
%sql

MERGE INTO FACT_LOAN_APPLICATION_SNAPSHOT T
USING CDC_UPDATES S
ON T.APP_ID = S.APP_ID

WHEN MATCHED THEN 
    UPDATE 
        SET T.APPROVED_DATE=S.APPROVED_DATE,
            T.DISBURSED_DATE=S.DISBURSED_DATE
WHEN NOT MATCHED THEN 
    INSERT(APP_ID,APPROVED_DATE,DISBURSED_DATE)
    VALUES(S.APP_ID,S.APPROVED_DATE,S.DISBURSED_DATE);


    
    

In [0]:
%sql
SELECT * FROM FACT_LOAN_APPLICATION_SNAPSHOT
WHERE APP_ID ='APP-9001';

In [0]:
%sql
SELECT  'DATE_SEQUENCE_CHK' AS CHECK_NAME,
        COUNT(*) AS INVALID_SEQUENCES,
        CASE WHEN COUNT(*) = 0 THEN 'PASSED' ELSE 'FAILED' END AS STATUS
FROM FACT_LOAN_APPLICATION_SNAPSHOT
WHERE SUBMITTED_DATE IS NOT NULL
AND UNDERWRITTEN_DATE IS NOT NULL
AND (
UNDERWRITTEN_DATE < SUBMITTED_DATE
OR (APPROVED_DATE IS NOT NULL
AND APPROVED_DATE < UNDERWRITTEN_DATE)
OR (DISBURSED_DATE IS NOT NULL
AND APPROVED_DATE IS NOT NULL
AND DISBURSED_DATE < APPROVED_DATE)
);

In [0]:
%sql
DESCRIBE HISTORY FACT_LOAN_APPLICATION_SNAPSHOT;

In [0]:
%sql
SELECT  APP_ID,
        APPROVED_DATE,
        DISBURSED_DATE
FROM FACT_LOAN_APPLICATION_SNAPSHOT VERSION AS OF 3
WHERE APP_ID='APP-9001';

In [0]:
%sql
OPTIMIZE FACT_CUSTOMER_FACILITY_ELIGIBILITY
ZORDER BY (CUSTOMER_ID,FACILITY_ID);

In [0]:
%sql
DESCRIBE HISTORY FACT_CUSTOMER_FACILITY_ELIGIBILITY;

In [0]:
%sql
SELECT  'FACT_CUSTOMER_FACILITY_ELIGIBILITY' AS TARGET_TABLE,
        'OPTIMIZED' AS CLUSTERING_STATUS;